In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import (classification_report, 
                              confusion_matrix,
                              roc_auc_score,
                              average_precision_score,
                              matthews_corrcoef,
                              cohen_kappa_score,
                              ConfusionMatrixDisplay)
import warnings 
warnings.filterwarnings('ignore')
import gdown
import pandas as pd

In [ ]:
file_id = "1XvkZeVyqoA6IGOYDaupyo0CljpRP90u7"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "Credit_card_data_set.csv", quiet=False)
df = pd.read_csv("Credit_card_data_set.csv")
df.head()

In [170]:
df.head(3)

,sn,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,21-06-2020 12:14,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,19-03-1968,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,21-06-2020 12:14,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",17-01-1990,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,21-06-2020 12:14,3.598220e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",21-10-1970,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0


In [171]:
df.isnull().sum()

sn                       0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

In [172]:
col = ['sn','cc_num','first', 'last','trans_num', 'unix_time','zip', 'state', 'street','merchant', 'city','job']
df = df.drop(columns = col, axis = 1)

In [173]:
df.head(3)

,trans_date_trans_time,category,amt,gender,lat,long,city_pop,dob,merch_lat,merch_long,is_fraud
0,21-06-2020 12:14,personal_care,2.86,M,33.9659,-80.9355,333497,19-03-1968,33.986391,-81.200714,0
1,21-06-2020 12:14,personal_care,29.84,F,40.3207,-110.4360,302,17-01-1990,39.450498,-109.960431,0
2,21-06-2020 12:14,health_fitness,41.28,F,40.6729,-73.5365,34496,21-10-1970,40.495810,-74.196111,0


In [174]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   trans_date_trans_time  555719 non-null  object 
 1   category               555719 non-null  object 
 2   amt                    555719 non-null  float64
 3   gender                 555719 non-null  object 
 4   lat                    555719 non-null  float64
 5   long                   555719 non-null  float64
 6   city_pop               555719 non-null  int64  
 7   dob                    555719 non-null  object 
 8   merch_lat              555719 non-null  float64
 9   merch_long             555719 non-null  float64
 10  is_fraud               555719 non-null  int64  
dtypes: float64(5), int64(2), object(4)
memory usage: 46.6+ MB


In [175]:
df['trans_date_trans_time'] =  pd.to_datetime(df['trans_date_trans_time'], dayfirst=True, format = 'mixed')
df['hour'] = df['trans_date_trans_time'].dt.hour
df['month'] = df['trans_date_trans_time'].dt.month
df['day'] = df['trans_date_trans_time'].dt.day
df.drop('trans_date_trans_time',axis =1, inplace = True)

In [176]:
df['dob'] = pd.to_datetime(df['dob'], dayfirst = True, format = 'mixed')
df['age'] = (pd.Timestamp.now() - df['dob']).dt.days//365
df.drop('dob',axis = 1)

,category,amt,gender,lat,long,city_pop,merch_lat,merch_long,is_fraud,hour,month,day,age
0,personal_care,2.86,M,33.9659,-80.9355,333497,33.986391,-81.200714,0,12,6,21,58
1,personal_care,29.84,F,40.3207,-110.4360,302,39.450498,-109.960431,0,12,6,21,36
2,health_fitness,41.28,F,40.6729,-73.5365,34496,40.495810,-74.196111,0,12,6,21,55
3,misc_pos,60.05,M,28.5697,-80.8191,54767,28.812398,-80.883061,0,12,6,21,38
4,travel,3.19,M,44.2529,-85.0170,1126,44.959148,-85.884734,0,12,6,21,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...
555714,health_fitness,43.77,M,40.4931,-91.8912,519,39.946837,-91.333331,0,23,12,31,60
555715,kids_pets,111.84,M,29.0393,-95.4401,28739,29.661049,-96.186633,0,23,12,31,26
555716,kids_pets,86.88,F,46.1966,-118.9017,3684,46.658340,-119.715054,0,23,12,31,44
555717,travel,7.99,M,44.6255,-116.4493,129,44.470525,-117.080888,0,23,12,31,60


In [177]:
category = list(df['category'].unique())
encoder = OneHotEncoder()
encoded = encoder.fit_transform(df[['category']]).toarray()
encoder_df = pd.DataFrame(encoded, columns = encoder.get_feature_names_out())
df1 = pd.concat([df,encoder_df], axis = 1)
encoded2 = encoder.fit_transform(df[['gender']]).toarray()
encoded_df2 = pd.DataFrame(encoded2, columns = encoder.get_feature_names_out())
df2= pd.concat([df1, encoded_df2], axis = 1)
df2.drop(columns = ['category', 'gender'],axis = 1, inplace = True)
df2.head(3)

,amt,lat,long,city_pop,dob,merch_lat,merch_long,is_fraud,hour,month,...,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_F,gender_M
0,2.86,33.9659,-80.9355,333497,1968-03-19,33.986391,-81.200714,0,12,6,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,29.84,40.3207,-110.4360,302,1990-01-17,39.450498,-109.960431,0,12,6,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,41.28,40.6729,-73.5365,34496,1970-10-21,40.495810,-74.196111,0,12,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [178]:
cols = (df2.columns.tolist())
clm = [col for col in cols if col!= 'is_fraud']
clm.append('is_fraud')
df= df2[clm]
df.head(3)

,amt,lat,long,city_pop,dob,merch_lat,merch_long,hour,month,day,...,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_F,gender_M,is_fraud
0,2.86,33.9659,-80.9355,333497,1968-03-19,33.986391,-81.200714,12,6,21,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
1,29.84,40.3207,-110.4360,302,1990-01-17,39.450498,-109.960431,12,6,21,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
2,41.28,40.6729,-73.5365,34496,1970-10-21,40.495810,-74.196111,12,6,21,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0


In [179]:
df['is_fraud'].value_counts(normalize = True)*100

is_fraud
0    99.614014
1     0.385986
Name: proportion, dtype: float64

In [180]:
df.drop(columns = ['dob'],axis = 1, inplace = True)

In [181]:
X = df.drop('is_fraud', axis = 1)
y = df['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, 
                                                    random_state = 42, 
                                                    stratify = y)
print(y_train.value_counts(normalize = True)*100)
print(y_test.value_counts(normalize = True)*100)

is_fraud
0    99.614013
1     0.385987
Name: proportion, dtype: float64
is_fraud
0    99.614014
1     0.385986
Name: proportion, dtype: float64


In [182]:
sm = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print('Shape before SMOTE :'  , X_train.shape)
print('Shape after SMOTE:', X_train_res.shape)
print('Target value count before SMOTE :', y_train.value_counts())
print(y_train.value_counts(normalize = True)*100)

print('Target value count after SMOTE:', y_train_res.value_counts())
print(y_train_res.value_counts(normalize = True)*100)

Shape before SMOTE : (444575, 26)
Shape after SMOTE: (487144, 26)
Target value count before SMOTE : is_fraud
0    442859
1      1716
Name: count, dtype: int64
is_fraud
0    99.614013
1     0.385987
Name: proportion, dtype: float64
Target value count after SMOTE: is_fraud
0    442859
1     44285
Name: count, dtype: int64
is_fraud
0    90.909259
1     9.090741
Name: proportion, dtype: float64


In [183]:
model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_rf.fit(X_train_res, y_train_res)

y_pred_rf = model_rf.predict(X_test)

In [184]:
model_xg = XGBClassifier(
    scale_pos_weight=10,  
    random_state=42,
    n_jobs=-1
)
model_xg.fit(X_train_res, y_train_res)
y_pred_xg = model.predict(X_test)


In [185]:
model_lgbm = LGBMClassifier(class_weight='balanced',
                             random_state=42, n_jobs=-1)
model_lgbm.fit(X_train_res, y_train_res)
y_pred_lgbm  = model_lgbm.predict(X_test)

[LightGBM] [Info] Number of positive: 44285, number of negative: 442859
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053591 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5747
[LightGBM] [Info] Number of data points in the train set: 487144, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


In [186]:
print('\nRandom Forest classifier\n')
print("ROC-AUC Score      :", roc_auc_score(y_test, y_pred_rf))
print("Avg Precision Score:", average_precision_score(y_test, y_pred_rf))
print("Matthews Corr Coef :", matthews_corrcoef(y_test, y_pred_rf))
print("Cohen Kappa Score  :", cohen_kappa_score(y_test, y_pred_rf))

print('\nXG boost classifier\n')

print("ROC-AUC Score      :", roc_auc_score(y_test, y_pred_xg))
print("Avg Precision Score:", average_precision_score(y_test, y_pred_xg))
print("Matthews Corr Coef :", matthews_corrcoef(y_test, y_pred_xg))
print("Cohen Kappa Score  :", cohen_kappa_score(y_test, y_pred_xg))


print('\nlgm classifier\n')
print("ROC-AUC Score      :", roc_auc_score(y_test, y_pred_lgbm))
print("Avg Precision Score:", average_precision_score(y_test, y_pred_lgbm))
print("Matthews Corr Coef :", matthews_corrcoef(y_test, y_pred_lgbm))
print("Cohen Kappa Score  :", cohen_kappa_score(y_test, y_pred_lgbm))


Random Forest classifier

ROC-AUC Score      : 0.8844663533188123
Avg Precision Score: 0.700191435642742
Matthews Corr Coef : 0.835668937265941
Cohen Kappa Score  : 0.83274153899635

XG boost classifier

ROC-AUC Score      : 0.9549974119273672
Avg Precision Score: 0.6494602364408836
Matthews Corr Coef : 0.8048535826932824
Cohen Kappa Score  : 0.7987187579086665

lgm classifier

ROC-AUC Score      : 0.9630659454802524
Avg Precision Score: 0.444698545370598
Matthews Corr Coef : 0.6650526158074793
Cohen Kappa Score  : 0.6294393831859815


In [187]:
c1 = y_test.tolist()
c_rf = y_pred_rf.tolist()
c_xg = y_pred_xg.tolist()
c_lgbm = y_pred_lgbm.tolist()
df3 = pd.DataFrame(data = {'is_fraud': c1, 'is_fraud_rf': c_rf,'is_fraud_xg': c_xg,'is_fraud_lgbm': c_lgbm})

X_test = X_test.reset_index(drop=True)
df3 = df3.reset_index(drop=True)
df_final = pd.concat([X_test, df3], axis=1)

In [188]:
df_final.head(3)

,amt,lat,long,city_pop,merch_lat,merch_long,hour,month,day,age,...,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_F,gender_M,is_fraud,is_fraud_rf,is_fraud_xg,is_fraud_lgbm
0,96.48,39.9347,-86.1633,910148,40.412801,-87.156553,14,12,29,38,...,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0,0
1,2.49,48.6669,-96.5969,140,47.876848,-96.159753,1,8,14,84,...,0.0,0.0,1.0,0.0,0.0,1.0,0,0,0,0
2,67.83,44.0385,-123.0614,191096,44.701304,-122.133886,20,12,4,62,...,0.0,0.0,0.0,0.0,1.0,0.0,0,0,0,0


In [ ]:
df_final.to_csv("Original vs predicted.csv")